<a href="https://colab.research.google.com/github/HariniRJ/ai-lab/blob/main/week3_a_1BM24CS364_HARINIRJ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import heapq
import copy

class PuzzleNode:
    def __init__(self, state, parent=None, action=None, g=0, h=0):
        self.state = state      # 3x3 list representation
        self.parent = parent    # Link to parent node
        self.action = action    # Action taken to reach this state ('L', 'U', 'R', 'D')
        self.g = g              # Depth / path cost from initial state
        self.h = h              # Heuristic cost estimation
        self.f = g + h          # Total cost evaluation function

    # Custom comparison operator for the priority queue (heapq)
    def __lt__(self, other):
        return self.f < other.f

# Helper function to find the row and column coordinates of the blank space (0)
def find_blank(state):
    for r in range(3):
        for c in range(3):
            if state[r][c] == 0:  # 0 represents the empty/blank space
                return r, c
    return None

# HEURISTIC 1: Number of misplaced tiles (Case 1)
def misplaced_tiles_heuristic(state, goal_state):
    misplaced = 0
    for r in range(3):
        for c in range(3):
            # Check if tile is misplaced (we ignore the blank space '0')
            if state[r][c] != 0 and state[r][c] != goal_state[r][c]:
                misplaced += 1
    return misplaced

# HEURISTIC 2: Manhattan Distance (Case 2)
def manhattan_distance_heuristic(state, goal_state):
    # Map each number to its goal row and column index
    goal_positions = {}
    for r in range(3):
        for c in range(3):
            goal_positions[goal_state[r][c]] = (r, c)

    distance = 0
    for r in range(3):
        for c in range(3):
            tile = state[r][c]
            if tile != 0:  # Ignore the blank space
                goal_r, goal_c = goal_positions[tile]
                distance += abs(r - goal_r) + abs(c - goal_c)
    return distance

# Generate valid next moves from the current node
def get_neighbors(node, goal_state, heuristic_type):
    neighbors = []
    r, c = find_blank(node.state)

    # Standard puzzle moves defined by the blank tile moving directions
    # 'L' -> Blank moves left (Tile moves right)
    # 'U' -> Blank moves up (Tile moves down)
    # 'R' -> Blank moves right (Tile moves left)
    # 'D' -> Blank moves down (Tile moves up)
    moves = {
        'L': (0, -1),
        'U': (-1, 0),
        'R': (0, 1),
        'D': (1, 0)
    }

    for direction, (dr, dc) in moves.items():
        new_r, new_c = r + dr, c + dc
        if 0 <= new_r < 3 and 0 <= new_c < 3:
            # Create a deep copy of the board to modify it safely
            new_state = copy.deepcopy(node.state)
            # Swap positions
            new_state[r][c], new_state[new_r][new_c] = new_state[new_r][new_c], new_state[r][c]

            # Calculate heuristic values based on configuration
            if heuristic_type == 'misplaced':
                h = misplaced_tiles_heuristic(new_state, goal_state)
            else:
                h = manhattan_distance_heuristic(new_state, goal_state)

            # Depth g(n) increases by 1 for each transition step
            neighbor_node = PuzzleNode(new_state, parent=node, action=direction, g=node.g + 1, h=h)
            neighbors.append(neighbor_node)

    return neighbors

# Core A* Search Algorithm
def a_star_search(initial_state, goal_state, heuristic_type='misplaced'):
    # Convert states to a tuple format to make them hashable for the visited set
    def state_to_tuple(state):
        return tuple(tuple(row) for row in state)

    # Calculate initial heuristic value
    if heuristic_type == 'misplaced':
        initial_h = misplaced_tiles_heuristic(initial_state, goal_state)
    else:
        initial_h = manhattan_distance_heuristic(initial_state, goal_state)

    start_node = PuzzleNode(initial_state, g=0, h=initial_h)

    # Frontier: priority queue ordered by ascending (g + h)
    frontier = []
    heapq.heappush(frontier, start_node)

    # Tracking closed/visited list to prevent infinite loop cycles
    visited = set()

    print(f"--- Running A* Search with {heuristic_type.upper()} heuristic ---")

    while frontier:
        current_node = heapq.heappop(frontier)

        # Base goal condition check
        if current_node.state == goal_state:
            return current_node  # Found the path to the solution node

        state_tuple = state_to_tuple(current_node.state)
        if state_tuple in visited:
            continue
        visited.add(state_tuple)

        # Expand neighbors
        for neighbor in get_neighbors(current_node, goal_state, heuristic_type):
            if state_to_tuple(neighbor.state) not in visited:
                heapq.heappush(frontier, neighbor)

    return None  # Failure case

# Helper function to print out a readable 3x3 layout matrix
def print_matrix(state):
    for row in state:
        print(" ".join(str(x) if x != 0 else "_" for x in row))
    print()

# Trace back paths through parent objects to display step progression output
def print_solution(solution_node):
    path = []
    current = solution_node
    while current:
        path.append(current)
        current = current.parent
    path.reverse()

    print(f"Goal Reached! Total Steps (Depth): {solution_node.g}\n")
    for step, node in enumerate(path):
        if node.action:
            print(f"Step {step}: Move Blank '{node.action}'")
        else:
            print(f"Step {step}: Initial State")
        print(f"g(n) = {node.g}, h(n) = {node.h}, f(n) = {node.f}")
        print_matrix(node.state)

# --- EXECUTION PROGRAM WITH CASE STUDY VALUES FROM SLIDES ---
if __name__ == "__main__":
    # In the matrix representation: 0 acts as the physical empty/blank space
    initial = [
        [2, 8, 3],
        [1, 6, 4],
        [7, 0, 5]
    ]

    goal = [
        [1, 2, 3],
        [8, 0, 4],
        [7, 6, 5]
    ]

    # Execute CASE 1: Misplaced Tiles Heuristic
    result_misplaced = a_star_search(initial, goal, heuristic_type='misplaced')
    if result_misplaced:
        print_solution(result_misplaced)
    else:
        print("No solution found using misplaced tiles heuristic.")

    print("="*40)

    # Execute CASE 2: Manhattan Distance Heuristic
    result_manhattan = a_star_search(initial, goal, heuristic_type='manhattan')
    if result_manhattan:
        print_solution(result_manhattan)
    else:
        print("No solution found using Manhattan distance heuristic.")

--- Running A* Search with MISPLACED heuristic ---
Goal Reached! Total Steps (Depth): 5

Step 0: Initial State
g(n) = 0, h(n) = 4, f(n) = 4
2 8 3
1 6 4
7 _ 5

Step 1: Move Blank 'U'
g(n) = 1, h(n) = 3, f(n) = 4
2 8 3
1 _ 4
7 6 5

Step 2: Move Blank 'U'
g(n) = 2, h(n) = 3, f(n) = 5
2 _ 3
1 8 4
7 6 5

Step 3: Move Blank 'L'
g(n) = 3, h(n) = 2, f(n) = 5
_ 2 3
1 8 4
7 6 5

Step 4: Move Blank 'D'
g(n) = 4, h(n) = 1, f(n) = 5
1 2 3
_ 8 4
7 6 5

Step 5: Move Blank 'R'
g(n) = 5, h(n) = 0, f(n) = 5
1 2 3
8 _ 4
7 6 5

--- Running A* Search with MANHATTAN heuristic ---
Goal Reached! Total Steps (Depth): 5

Step 0: Initial State
g(n) = 0, h(n) = 5, f(n) = 5
2 8 3
1 6 4
7 _ 5

Step 1: Move Blank 'U'
g(n) = 1, h(n) = 4, f(n) = 5
2 8 3
1 _ 4
7 6 5

Step 2: Move Blank 'U'
g(n) = 2, h(n) = 3, f(n) = 5
2 _ 3
1 8 4
7 6 5

Step 3: Move Blank 'L'
g(n) = 3, h(n) = 2, f(n) = 5
_ 2 3
1 8 4
7 6 5

Step 4: Move Blank 'D'
g(n) = 4, h(n) = 1, f(n) = 5
1 2 3
_ 8 4
7 6 5

Step 5: Move Blank 'R'
g(n) = 5, h(n) = 0, 